# Budujemy agenta AI w Pythonie

Krok po kroku zbudujemy agenta IT helpdesk, ktory:
1. Wywoluje model LLM przez API (OpenRouter)
2. Ma **narzedzia** (tools) zdefiniowane jako JSON Schema
3. Sam decyduje, ktore narzedzie wywolac (native tool calling)
4. Pamięta kontekst rozmowy (memory)
5. Wykonuje **akcje** — nie tylko odpowiada, ale tez tworzy zgloszenia

**Czas:** ~60 minut | **GPU NIE jest wymagane!**

| Krok | Temat | Czas |
|------|-------|------|
| 0 | Setup — polaczenie z API | ~3 min |
| 1 | Podstawowe wywolanie LLM | ~10 min |
| 2 | Definiowanie narzedzi (tools) | ~15 min |
| 3 | Petla agenta (agent loop) | ~15 min |
| 4 | Pamiec (memory) | ~10 min |
| 5 | Pelne demo | ~10 min |

## Krok 0: Setup

Uzywamy **OpenRouter** — uniwersalnego API do wielu modeli LLM. Nie potrzebujemy GPU, nie pobieramy modelu. Wystarczy klucz API.

### Jak ustawic klucz API:
- **Colab:** dodaj `OPENROUTER_API_KEY` w Secrets (ikona klucza po lewej)
- **Lokalnie:** `export OPENROUTER_API_KEY="sk-or-..."`
- **Jesli nie masz klucza:** notebook zapyta o niego interaktywnie

In [ ]:
import requests
import json
import os
import pandas as pd
from getpass import getpass

# === Konfiguracja API ===
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "meta-llama/llama-3.1-8b-instruct"

# Zaladuj klucz API: os.environ -> Colab Secrets -> getpass
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")

if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("OPENROUTER_API_KEY")
        print("Klucz API zaladowany z Colab Secrets")
    except Exception:
        pass

if not API_KEY:
    API_KEY = getpass("Wklej klucz OpenRouter API: ")

# Test polaczenia
USE_API = False
if API_KEY:
    test_resp = requests.post(
        url=OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": "Powiedz OK"}],
            "max_tokens": 5,
        },
        timeout=15,
    )
    if test_resp.ok:
        USE_API = True
        print(f"Model: {MODEL}")
        print(f"API: dziala!")
    else:
        print(f"API error {test_resp.status_code}: {test_resp.text[:200]}")
        print("Przelaczam na tryb fallback (mock responses)")
else:
    print("Brak klucza API — tryb fallback (mock responses)")
    print("Notebook bedzie dzialal, ale odpowiedzi beda symulowane.")

---
## Krok 1: Podstawowe wywolanie LLM (~10 min)

Zaczynamy od najprostszej rzeczy: wysylamy pytanie do modelu i dostajemy odpowiedz.

### Wytlumaczenie:
Kazdy agent AI zaczyna od jednej umiejetnosci: wywolania modelu. To jest nasz fundament.

Funkcja `call_llm()` to "mozg" agenta. Uzywamy formatu **chat completions** — lista wiadomosci z rolami (`system`, `user`, `assistant`). Ten sam format uzywaja OpenAI, Anthropic, Google i inne.

### Cwiczenie:
Zmien `system_prompt` na wlasny i sprawdz efekt.

In [ ]:
def call_llm(messages, tools=None, temperature=0):
    """Wywolaj LLM i zwroc pelny obiekt wiadomosci.

    Args:
        messages: lista wiadomosci [{"role": ..., "content": ...}]
        tools: opcjonalna lista narzedzi (JSON Schema)
        temperature: 0 = deterministyczny, 1 = kreatywny

    Returns:
        dict — pelny obiekt message (content + ewentualne tool_calls)
    """
    if not USE_API:
        # Fallback: zwroc mock response
        user_msg = messages[-1]["content"] if messages else ""
        return {"role": "assistant", "content": f"[FALLBACK] Odpowiedz na: {user_msg[:80]}..."}

    print("Czekam na model...", end=" ")

    payload = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
    }
    if tools:
        payload["tools"] = tools

    resp = requests.post(
        url=OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=60,
    )

    if not resp.ok:
        print(f"Blad API: {resp.status_code}")
        return {"role": "assistant", "content": f"Blad API: {resp.text[:200]}"}

    message = resp.json()["choices"][0]["message"]
    print("OK")
    return message


print("Funkcja call_llm() zdefiniowana")
print("Zwraca pelny obiekt message — nie tylko tekst, ale tez ewentualne tool_calls")

In [ ]:
# Test: proste pytanie po polsku
SYSTEM_PROMPT = (
    "Jestes asystentem IT helpdesk w polskiej firmie. "
    "Odpowiadasz krotko, rzeczowo, po polsku."
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Czym zajmuje sie helpdesk IT? Odpowiedz w 2 zdaniach."},
]

response = call_llm(messages)
print(f"\nOdpowiedz modelu:\n{response['content']}")

---
## Krok 2: Definiowanie narzedzi (Tools) (~15 min)

Agent to nie chatbot. Agent ma **narzedzia** i sam decyduje, ktorego uzyc.

### Wytlumaczenie:
Definiujemy 4 narzedzia jako funkcje Pythona + ich opisy jako **JSON Schema**. Model dostaje te opisy i sam decyduje:
- Czy uzyc narzedzia?
- Ktorego?
- Z jakimi parametrami?

To jest **native tool calling** — model zwraca strukturalne wywolanie narzedzia, nie wolny tekst do parsowania.

**Klasyfikacja NIE jest narzedziem** — to rozumowanie, ktore model robi natywnie. Gdy tworzy zgloszenie, sam decyduje o kategorii i priorytecie.

### 4 narzedzia:
| # | Narzedzie | Co robi | Punkt dydaktyczny |
|---|-----------|---------|-------------------|
| 1 | `search_knowledge_base` | Szuka w bazie wiedzy IT | Narzedzia daja dostep do wiedzy (symulacja RAG) |
| 2 | `get_ticket_status` | Sprawdza status zgloszenia | Narzedzia lacza sie z systemami zewnetrznymi |
| 3 | `query_ticket_stats` | Statystyki zgloszen (pandas) | Narzedzia daja dostep do danych |
| 4 | `create_ticket` | Tworzy nowe zgloszenie | Narzedzia moga wykonywac AKCJE (zapis) |

### Cwiczenie:
Dodaj 5. narzedzie: `escalate_ticket(ticket_id, reason)` — oznacza zgloszenie jako pilne.

In [ ]:
# === Dane: 24 zgloszenia (6 na kategorie) — notebook jest samodzielny ===
ticket_data = [
    # Sprzet
    {"id": "T-001", "text": "Laptop nie wlacza sie po aktualizacji BIOS", "category": "Sprzet", "priority": "P1", "status": "Otwarty", "assignee": "Zespol Hardware"},
    {"id": "T-002", "text": "Monitor migocze przy rozdzielczosci 4K", "category": "Sprzet", "priority": "P3", "status": "W trakcie", "assignee": "Zespol Hardware"},
    {"id": "T-003", "text": "Klawiatura bezprzewodowa nie reaguje", "category": "Sprzet", "priority": "P3", "status": "Zamkniety", "assignee": "Zespol Hardware"},
    {"id": "T-004", "text": "Stacja dokujaca nie wykrywa monitorow", "category": "Sprzet", "priority": "P2", "status": "Otwarty", "assignee": "Zespol Hardware"},
    {"id": "T-005", "text": "Bateria laptopa trzyma tylko 30 minut", "category": "Sprzet", "priority": "P2", "status": "W trakcie", "assignee": "Zespol Hardware"},
    {"id": "T-006", "text": "Drukarka zacina papier", "category": "Sprzet", "priority": "P3", "status": "Zamkniety", "assignee": "Zespol Hardware"},
    # Siec
    {"id": "T-007", "text": "VPN rozlacza sie co 10 minut", "category": "Siec", "priority": "P1", "status": "Otwarty", "assignee": "Zespol Sieciowy"},
    {"id": "T-008", "text": "WiFi nie dziala na 2. pietrze", "category": "Siec", "priority": "P2", "status": "W trakcie", "assignee": "Zespol Sieciowy"},
    {"id": "T-009", "text": "Drukarka sieciowa nie odpowiada", "category": "Siec", "priority": "P2", "status": "Otwarty", "assignee": "Zespol Sieciowy"},
    {"id": "T-010", "text": "Brak dostepu do serwera plikow", "category": "Siec", "priority": "P1", "status": "W trakcie", "assignee": "Zespol Sieciowy"},
    {"id": "T-011", "text": "Internet wolny w calym biurze", "category": "Siec", "priority": "P1", "status": "Otwarty", "assignee": "Zespol Sieciowy"},
    {"id": "T-012", "text": "Nie mozna polaczyc sie z VPN z domu", "category": "Siec", "priority": "P2", "status": "Zamkniety", "assignee": "Zespol Sieciowy"},
    # Oprogramowanie
    {"id": "T-013", "text": "Excel zawiesza sie przy duzych plikach", "category": "Oprogramowanie", "priority": "P2", "status": "Otwarty", "assignee": "Zespol Software"},
    {"id": "T-014", "text": "Antywirus blokuje instalacje programu", "category": "Oprogramowanie", "priority": "P3", "status": "W trakcie", "assignee": "Zespol Software"},
    {"id": "T-015", "text": "System ERP nie generuje raportow", "category": "Oprogramowanie", "priority": "P1", "status": "Otwarty", "assignee": "Zespol Software"},
    {"id": "T-016", "text": "Aplikacja CRM zawiesza sie po aktualizacji", "category": "Oprogramowanie", "priority": "P2", "status": "W trakcie", "assignee": "Zespol Software"},
    {"id": "T-017", "text": "Outlook nie synchronizuje poczty", "category": "Oprogramowanie", "priority": "P2", "status": "Otwarty", "assignee": "Zespol Software"},
    {"id": "T-018", "text": "Teams nie laczy sie z kamera", "category": "Oprogramowanie", "priority": "P3", "status": "Zamkniety", "assignee": "Zespol Software"},
    # Konto
    {"id": "T-019", "text": "Nie moge zmienic hasla w systemie ERP", "category": "Konto", "priority": "P2", "status": "Otwarty", "assignee": "Zespol Kont"},
    {"id": "T-020", "text": "Konto zablokowane po 3 probach logowania", "category": "Konto", "priority": "P1", "status": "W trakcie", "assignee": "Zespol Kont"},
    {"id": "T-021", "text": "Brak dostepu do folderu sieciowego po zmianie hasla", "category": "Konto", "priority": "P2", "status": "Otwarty", "assignee": "Zespol Kont"},
    {"id": "T-022", "text": "Nowy pracownik potrzebuje konta w systemach", "category": "Konto", "priority": "P2", "status": "Otwarty", "assignee": "Zespol Kont"},
    {"id": "T-023", "text": "Nie moge zalogowac sie do poczty po urlopie", "category": "Konto", "priority": "P2", "status": "W trakcie", "assignee": "Zespol Kont"},
    {"id": "T-024", "text": "Potrzebuje dostepu do systemu CRM", "category": "Konto", "priority": "P3", "status": "Zamkniety", "assignee": "Zespol Kont"},
]

df_tickets = pd.DataFrame(ticket_data)

# Opcjonalnie: zaladuj wiekszy plik CSV jesli jest dostepny
try:
    df_large = pd.read_csv("large_tickets.csv")
    print(f"Zaladowano rowniez large_tickets.csv ({len(df_large)} zgloszen)")
except FileNotFoundError:
    df_large = None

print(f"Baza zgloszen: {len(df_tickets)} zgloszen")
print(f"Kategorie: {df_tickets['category'].value_counts().to_dict()}")
print(f"\nPrzykladowe zgloszenia:")
print(df_tickets[["id", "text", "category", "priority", "status"]].head(6).to_string(index=False))

In [ ]:
# === 4 narzedzia: funkcje Pythona ===

def search_knowledge_base(query):
    """Przeszukaj baze wiedzy IT (runbooki) po slowach kluczowych."""
    kb = {
        "drukarka": "RUNBOOK: Drukarki sieciowe (IP/WiFi) — jesli nie odpowiada, sprawdz polaczenie sieciowe. Problem sieciowy = kategoria Siec. Fizyczna usterka (zacina papier, brak tonera) = Sprzet.",
        "wifi": "RUNBOOK: Problemy z WiFi — brak polaczenia, slaby sygnal, rozlaczanie sie = Siec. Problemy z logowaniem do portalu WiFi = Konto.",
        "haslo": "RUNBOOK: Zmiana/reset hasla — niezaleznie od systemu (ERP, CRM, poczta), root cause to zarzadzanie kontem = Konto. Po zmianie hasla wymagana synchronizacja z Active Directory.",
        "vpn": "RUNBOOK: VPN — problemy z polaczeniem, rozlaczanie, certyfikaty = Siec. Sprawdz: 1) certyfikat wazny? 2) firewall? 3) split tunneling?",
        "antywirus": "RUNBOOK: Antywirus blokuje program — dodaj wyjatki w ustawieniach. To konflikt oprogramowania = Oprogramowanie. Nie wylaczaj antywirusa!",
        "erp": "RUNBOOK: System ERP — bledy raportow, zawieszanie sie = Oprogramowanie. Problemy z logowaniem/haslem = Konto. Sprawdz wersje i logi.",
        "monitor": "RUNBOOK: Monitor — migotanie, brak obrazu, artefakty = Sprzet. Sprawdz kabel, sterowniki GPU, rozdzielczosc.",
        "laptop": "RUNBOOK: Laptop — nie wlacza sie, wolno dziala, przegrzewa sie = Sprzet. Sprawdz BIOS, baterie, wentylator. Po aktualizacji BIOS: reset CMOS.",
    }
    query_lower = query.lower()
    results = [v for k, v in kb.items() if k in query_lower]
    return " | ".join(results) if results else "Brak artykulow w bazie wiedzy dla tego zapytania."


def get_ticket_status(ticket_id):
    """Sprawdz status zgloszenia po jego ID (np. T-001)."""
    match = df_tickets[df_tickets["id"] == ticket_id]
    if match.empty:
        return json.dumps({"error": f"Zgloszenie {ticket_id} nie znalezione"}, ensure_ascii=False)
    row = match.iloc[0]
    return json.dumps({
        "id": row["id"],
        "text": row["text"],
        "status": row["status"],
        "category": row["category"],
        "priority": row["priority"],
        "assignee": row["assignee"],
    }, ensure_ascii=False, indent=2)


def query_ticket_stats(action, category=None):
    """Statystyki zgloszen: count, top, sample. Opcjonalny filtr po kategorii."""
    data = df_tickets
    if category:
        data = df_tickets[df_tickets["category"] == category]
        if data.empty:
            return f"Brak zgloszen w kategorii '{category}'. Dostepne: {list(df_tickets['category'].unique())}"

    if action == "count":
        if category:
            return f"Kategoria '{category}': {len(data)} zgloszen"
        counts = df_tickets["category"].value_counts().to_dict()
        return f"Laczna liczba zgloszen: {len(df_tickets)}. Rozklad: {counts}"

    elif action == "top":
        top_cats = df_tickets["category"].value_counts()
        lines = [f"  {cat}: {cnt} zgloszen" for cat, cnt in top_cats.items()]
        return "Najczestsze kategorie:\n" + "\n".join(lines)

    elif action == "sample":
        samples = data.sample(min(3, len(data)))
        lines = [f"  [{r['id']}] [{r['category']}] {r['text']}" for _, r in samples.iterrows()]
        return "Przykladowe zgloszenia:\n" + "\n".join(lines)

    return f"Nieznana akcja: '{action}'. Dozwolone: count, top, sample"


def create_ticket(summary, category, priority):
    """Utworz nowe zgloszenie IT."""
    global df_tickets
    new_id = f"T-{len(df_tickets) + 1:03d}"
    new_row = pd.DataFrame([{
        "id": new_id,
        "text": summary,
        "category": category,
        "priority": priority,
        "status": "Otwarty",
        "assignee": "Nieprzypisany",
    }])
    df_tickets = pd.concat([df_tickets, new_row], ignore_index=True)
    return json.dumps({
        "status": "utworzono",
        "ticket_id": new_id,
        "summary": summary,
        "category": category,
        "priority": priority,
    }, ensure_ascii=False, indent=2)


# Rejestr narzedzi — mapowanie nazw na funkcje
TOOL_FUNCTIONS = {
    "search_knowledge_base": search_knowledge_base,
    "get_ticket_status": get_ticket_status,
    "query_ticket_stats": query_ticket_stats,
    "create_ticket": create_ticket,
}

print("4 narzedzia zdefiniowane:")
for name, func in TOOL_FUNCTIONS.items():
    print(f"  {name}: {func.__doc__}")

In [ ]:
# === JSON Schema — opisy narzedzi dla modelu ===
# Format OpenAI-compatible (uzywa go tez OpenRouter, Anthropic, Google)

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Przeszukaj baze wiedzy IT (runbooki, procedury) po slowach kluczowych. Uzyj gdy uzytkownik pyta o rozwiazanie problemu.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Slowa kluczowe do wyszukania, np. 'drukarka', 'vpn', 'haslo'",
                    }
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_ticket_status",
            "description": "Sprawdz status istniejacego zgloszenia IT po jego identyfikatorze (np. T-001, T-015).",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticket_id": {
                        "type": "string",
                        "description": "Identyfikator zgloszenia, np. T-001",
                    }
                },
                "required": ["ticket_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_ticket_stats",
            "description": "Pobierz statystyki zgloszen IT: liczbe zgloszen (count), najczestsze kategorie (top), lub przykladowe zgloszenia (sample). Opcjonalny filtr po kategorii.",
            "parameters": {
                "type": "object",
                "properties": {
                    "action": {
                        "type": "string",
                        "enum": ["count", "top", "sample"],
                        "description": "Rodzaj zapytania: count=ile zgloszen, top=najczestsze kategorie, sample=przykladowe zgloszenia",
                    },
                    "category": {
                        "type": "string",
                        "description": "Opcjonalny filtr kategorii: Sprzet, Siec, Oprogramowanie, Konto",
                    },
                },
                "required": ["action"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_ticket",
            "description": "Utworz nowe zgloszenie IT. Agent sam decyduje o kategorii i priorytecie na podstawie opisu problemu.",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {
                        "type": "string",
                        "description": "Krotki opis problemu",
                    },
                    "category": {
                        "type": "string",
                        "description": "Kategoria zgloszenia: Sprzet, Siec, Oprogramowanie lub Konto",
                    },
                    "priority": {
                        "type": "string",
                        "enum": ["P1", "P2", "P3", "P4"],
                        "description": "Priorytet: P1=krytyczny, P2=wysoki, P3=sredni, P4=niski",
                    },
                },
                "required": ["summary", "category", "priority"],
            },
        },
    },
]

# Pokaz przyklad schematu
print("Przyklad schematu narzedzia (create_ticket):")
print(json.dumps(TOOLS_SCHEMA[3], indent=2, ensure_ascii=False))

---
## Krok 3: Petla agenta (Agent Loop) (~15 min)

Serce agenta: model dostaje pytanie + liste narzedzi, decyduje czy i ktore wywolac.

### Wytlumaczenie:
Petla agenta dziala tak:

```
Uzytkownik → [pytanie] → LLM + narzedzia
                              |
                    ┌─── tool_calls? ───┐
                    │                   │
                   TAK                 NIE
                    │                   │
            Wykonaj narzedzie    Zwroc odpowiedz
                    │
            Dodaj wynik do wiadomosci
                    │
            Wywolaj LLM ponownie
                    │
            Zwroc koncowa odpowiedz
```

Kluczowe: **native tool_calls** — model zwraca strukturalne wywolanie, zero parsowania JSON z tekstu!

### Cwiczenie:
Dodaj obsluge sytuacji, gdy model wywola narzedzie, ktore nie istnieje. Co powinien zwrocic agent?

In [ ]:
AGENT_SYSTEM_PROMPT = (
    "Jestes agentem IT helpdesk w polskiej firmie. "
    "Masz dostep do narzedzi: bazy wiedzy, systemu zgloszen, statystyk i tworzenia zgloszen. "
    "Odpowiadasz po polsku, krotko i rzeczowo. "
    "Gdy uzytkownik prosi o utworzenie zgloszenia, sam decydujesz o kategorii i priorytecie na podstawie opisu problemu. "
    "Dostepne kategorie: Sprzet, Siec, Oprogramowanie, Konto. "
    "Priorytety: P1=krytyczny (system nie dziala), P2=wysoki (utrudnia prace), P3=sredni (mozna obejsc), P4=niski (kosmetyczny)."
)


def execute_tool(name, arguments):
    """Wykonaj narzedzie po nazwie z podanymi argumentami."""
    func = TOOL_FUNCTIONS.get(name)
    if not func:
        return f"Nieznane narzedzie: {name}"
    args = json.loads(arguments)
    return func(**args)


def agent_chat(user_message, messages=None):
    """Petla agenta: pytanie -> (opcjonalnie tool_calls) -> odpowiedz."""
    if messages is None:
        messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}]

    # 1. Dodaj wiadomosc uzytkownika
    messages.append({"role": "user", "content": user_message})

    # 2. Wywolaj LLM z narzedziami
    response = call_llm(messages, tools=TOOLS_SCHEMA)

    # 3. Sprawdz czy model chce uzyc narzedzia
    tool_calls = response.get("tool_calls")

    if tool_calls:
        # Model zdecydowal uzyc narzedzi
        messages.append(response)  # dodaj odpowiedz asystenta (z tool_calls)

        for tool_call in tool_calls:
            func_name = tool_call["function"]["name"]
            func_args = tool_call["function"]["arguments"]
            tool_id = tool_call["id"]

            print(f"  Narzedzie: {func_name}({func_args})")
            result = execute_tool(func_name, func_args)
            print(f"  Wynik: {result[:200]}")

            # Dodaj wynik narzedzia do wiadomosci
            messages.append({
                "role": "tool",
                "tool_call_id": tool_id,
                "content": str(result),
            })

        # 4. Wywolaj LLM ponownie — teraz z wynikami narzedzi
        final_response = call_llm(messages)
        messages.append(final_response)
        return final_response.get("content", ""), messages
    else:
        # Brak tool_calls — odpowiedz bezposrednia
        messages.append(response)
        return response.get("content", ""), messages


# === Test: 3 zapytania uzywajace roznych narzedzi ===

print("=" * 60)
print("Test 1: Szukanie w bazie wiedzy")
print("=" * 60)
answer, _ = agent_chat("Mam problem z drukarka sieciowa — co moge sprawdzic?")
print(f"\nAgent: {answer}")

print(f"\n{'=' * 60}")
print("Test 2: Status zgloszenia")
print("=" * 60)
answer, _ = agent_chat("Jaki jest status zgloszenia T-007?")
print(f"\nAgent: {answer}")

print(f"\n{'=' * 60}")
print("Test 3: Statystyki zgloszen")
print("=" * 60)
answer, _ = agent_chat("Ile mamy otwartych zgloszen? Pokaz statystyki.")
print(f"\nAgent: {answer}")

In [ ]:
# === Kluczowy moment: tworzenie zgloszenia ===
# Agent sam decyduje o kategorii i priorytecie!

print("=" * 60)
print("Test 4: Utworzenie zgloszenia")
print("=" * 60)

answer, _ = agent_chat(
    "Utworz zgloszenie: laptop nie wlacza sie po aktualizacji BIOS, "
    "caly dzial nie moze pracowac"
)
print(f"\nAgent: {answer}")

print("\n" + "=" * 60)
print("Klasyfikacja to rozumowanie, nie narzedzie!")
print("Agent sam zdecydowal o kategorii i priorytecie.")
print("=" * 60)

# Pokaz aktualna baze zgloszen
print(f"\nBaza zgloszen po dodaniu ({len(df_tickets)} zgloszen):")
print(df_tickets[["id", "text", "category", "priority"]].tail(3).to_string(index=False))

---
## Krok 4: Pamiec (Memory) (~10 min)

Dotychczas kazde pytanie bylo niezalezne. Teraz dodajemy pamiec — agent pamięta, o czym rozmawialismy.

### Wytlumaczenie:
Pamiec to po prostu **lista wiadomosci**. Przy kazdym wywolaniu dodajemy cala historie do kontekstu. Model widzi poprzednie pytania i odpowiedzi, wiec moze odpowiadac na pytania typu "A co z tamtym zgloszeniem?" bez powtarzania.

W praktyce: pamiec = `messages` lista, ktora rosnie z kazda wymiana.

### Cwiczenie:
Dodaj limit pamieci: agent pamięta tylko ostatnie N wiadomosci. Dlaczego to wazne? (context window!)

In [ ]:
class ITAgent:
    """Agent IT helpdesk z pamiecia rozmowy."""

    def __init__(self):
        self.messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}]

    def chat(self, user_message):
        """Wyslij wiadomosc do agenta (z pamiecia kontekstu)."""
        answer, self.messages = agent_chat(user_message, self.messages)
        return answer

    def show_memory(self):
        """Pokaz pelna historie rozmowy."""
        print("Historia rozmowy:")
        for i, msg in enumerate(self.messages):
            role = msg.get("role", "?")
            if role == "system":
                print(f"  [{i}] SYSTEM: {msg['content'][:80]}...")
            elif role == "user":
                print(f"  [{i}] USER: {msg['content'][:80]}")
            elif role == "assistant":
                content = msg.get("content", "")
                tool_calls = msg.get("tool_calls")
                if tool_calls:
                    tools_used = [tc["function"]["name"] for tc in tool_calls]
                    print(f"  [{i}] ASSISTANT: [tool_calls: {', '.join(tools_used)}]")
                elif content:
                    print(f"  [{i}] ASSISTANT: {content[:80]}")
            elif role == "tool":
                print(f"  [{i}] TOOL: {msg['content'][:80]}")


# === Rozmowa wieloturowa ===
agent = ITAgent()

print("--- Tura 1 ---")
r = agent.chat("Jaki jest status zgloszenia T-007?")
print(f"Agent: {r}\n")

print("--- Tura 2 ---")
r = agent.chat("A co z T-001?")
print(f"Agent: {r}\n")

print("--- Tura 3 ---")
r = agent.chat("Przeszukaj baze wiedzy — co moge zrobic z problemem VPN?")
print(f"Agent: {r}\n")

print("--- Tura 4 ---")
r = agent.chat("Na podstawie tego co znalezlismy — utworz zgloszenie o tym problemie z VPN")
print(f"Agent: {r}\n")

print("=" * 60)
agent.show_memory()

---
## Krok 5: Pelne demo (~10 min)

Laczymy wszystko w jeden scenariusz: uzytkownik zglasza problem, agent szuka w bazie wiedzy, tworzy zgloszenie, sprawdza statystyki.

In [ ]:
# === Pelny scenariusz demo ===
demo_agent = ITAgent()

demo_queries = [
    "Czesc, mam problem — drukarka sieciowa na 2. pietrze nie odpowiada od rana.",
    "Przeszukaj baze wiedzy, co moge sprawdzic?",
    "OK, utworz zgloszenie o tym problemie.",
    "Ile mamy teraz zgloszen w kategorii Siec?",
]

for i, q in enumerate(demo_queries, 1):
    print(f"\n{'=' * 60}")
    print(f"Uzytkownik [{i}]: {q}")
    print("-" * 60)
    answer = demo_agent.chat(q)
    print(f"\nAgent: {answer}")

print(f"\n{'=' * 60}")
print("\nPelna historia rozmowy:")
demo_agent.show_memory()

In [ ]:
# === Opcjonalnie: interaktywna petla (odkomentuj aby uzywac) ===

# interactive_agent = ITAgent()
# print("Agent IT Helpdesk gotowy! Wpisz 'quit' aby zakonczyc.")
# print("Przykladowe pytania:")
# print("  - Jaki jest status zgloszenia T-010?")
# print("  - Mam problem z VPN, co moge zrobic?")
# print("  - Utworz zgloszenie: Excel zawiesza sie przy duzych plikach")
# print("  - Ile mamy zgloszen w kategorii Sprzet?")
# print()
#
# while True:
#     user_input = input("Ty: ")
#     if user_input.lower() in ["quit", "exit", "q"]:
#         print("Do widzenia!")
#         break
#     answer = interactive_agent.chat(user_input)
#     print(f"Agent: {answer}\n")

---
## Podsumowanie: Anatomia agenta AI

Wlasnie zbudowaliscie agenta AI od zera. Zobaczmy co ma w srodku:

```
┌─────────────────────────────────────────────┐
│           AGENT IT HELPDESK                 │
├─────────────────────────────────────────────┤
│ Mozg:      LLM (Llama 3.1 8B via API)      │
│ Narzedzia: search_kb, get_status,           │
│            query_stats, create_ticket        │
│ Pamiec:    lista wiadomosci (messages)       │
│ Decyzje:   native tool_calls (JSON Schema)   │
│ Akcje:     odczyt I zapis (create_ticket)    │
└─────────────────────────────────────────────┘
```

| Krok | Komponent | Co robi |
|------|-----------|--------|
| 1 | `call_llm()` | Bazowe wywolanie modelu przez API |
| 2 | Tools (JSON Schema) | Definicje narzedzi — model wie co moze robic |
| 3 | Agent loop | Petla: pytanie -> tool_calls -> wynik -> odpowiedz |
| 4 | Memory | Lista wiadomosci — agent pamięta kontekst |
| 5 | Full agent | Wszystko razem: rozmowa, narzedzia, pamiec, akcje |

### 5 kluczowych lekcji:

1. **Agent = LLM + narzedzia + pamiec.** Bez narzedzi to chatbot. Bez pamieci to amnezja.
2. **Native tool calling > parsowanie tekstu.** Model zwraca `tool_calls`, nie tekst do regexowania.
3. **Klasyfikacja to rozumowanie, nie narzedzie.** Agent sam decyduje o kategorii — to jego inteligencja.
4. **Narzedzia moga CZYTAC i PISAC.** `search_knowledge_base` czyta, `create_ticket` pisze.
5. **Pamiec to lista wiadomosci.** Proste, ale potezne — dlatego context window jest tak wazny.

### Cwiczenie koncowe:
Rozszerz agenta o nowe narzedzie: `escalate_ticket(ticket_id, reason)` ktore:
1. Zmienia priorytet zgloszenia na P1
2. Dodaje komentarz z powodem eskalacji
3. Zwraca potwierdzenie

Pamietaj: potrzebujesz zarowno **funkcji Pythona** jak i **JSON Schema**!